# DIMER Notebook: Document-Type Classification with DiT
## RVL-CDIP with Microsoft Document Image Transformer (DiT)

**Notebook profile:** `TASK-INFERENCE`  
**Pedagogical mode:** `WORKSHOP`  
**DIMER Notebook Specification:** `2.1`  
**Standalone:** yes  
**Task:** whole-page document-type classification  
**Canonical model:** `microsoft/dit-base-finetuned-rvlcdip`  
**Classes:** 16  
**Adaptation:** none

This notebook treats each document page as an image and predicts one of the 16 RVL-CDIP document types.

The notebook covers:

- immutable model acquisition and an explicit `.bin → SafeTensors` candidate conversion boundary;
- balanced RVL-CDIP-derived evaluation sampling;
- accuracy, macro F1 and top-3 accuracy;
- per-class metrics and confusion analysis;
- high-confidence and low-margin errors;
- rotation, resolution and crop robustness;
- score/margin/entropy interpretation;
- machine-readable outputs and provenance; and
- optional BYOD page classification.

> **Candidate status.** The upstream model repository currently exposes a PyTorch pickle checkpoint on its pinned revision. This notebook verifies the exact source SHA-256, loads the state dict with `weights_only=True`, converts it locally to SafeTensors, proves logit parity, and performs the remainder of inference from the converted representation. A future DIMER-hosted release should ship the already-converted SafeTensors snapshot and remove the source-conversion step from the user path.

## How to use this notebook

**Who it is for.** Learners who can run Python cells in Colab/Jupyter and are new to image classification or document intelligence.

**Runtime.** Use the documented GPU runtime for the canonical DiT evaluation.

**How to run it.**
1. Select the documented runtime/accelerator.
2. Choose **Run all** for the canonical path; leave the default settings unchanged on your first pass.
3. Read the explanatory markdown while the notebook runs.
4. Sections marked **Infrastructure** support reproducibility, model acquisition, or orchestration. Run those cells as written; understanding their implementation is not a learning objective.

### Task at a glance

`document page image → DiT classifier → fixed document-type label + score → confusion/robustness analysis`

### Roadmap

1. Understand the task and its input/output contract.
2. Inspect and validate the built-in data or inputs.
3. Establish the baseline/reference behavior.
4. Run the model or multi-model comparison.
5. Inspect errors, disagreements, robustness, and/or resource tradeoffs.
6. Try one controlled change and explain what changed.
7. Write an evidence-based conclusion; optionally continue with BYOD.

### What successful execution looks like

You should finish with a validated input/sample, the notebook's principal baseline/reference, model outputs and evaluation results, at least one diagnostic or qualitative comparison, and machine-readable results/provenance where supported. Exact values can vary slightly across supported runtimes; focus on the defined metrics and the observed pattern.

**Reading layers:** Core concepts explain page images, learned representations and fixed classes. Evaluation practice covers baselines, metrics and controlled transformations. Infrastructure cells install packages, verify weights and prepare data; their code is collapsed and may stay collapsed. For active learning, run section by section and write each prediction before revealing results. The optional activity below reads existing robustness results without changing canonical predictions.


## 1. Document classification is not OCR or Document QA

This notebook answers:

> **What kind of document is this page?**

It does not answer:

- What text is on the page?
- What field/value appears in the document?
- What does a table contain?
- What is the answer to a question about the page?

The classifier uses visual evidence from the whole rendered page and outputs one fixed RVL-CDIP class.

## 2. Why DiT?

Microsoft's Document Image Transformer (DiT) adapts a vision-transformer-style architecture to document images. The selected checkpoint is already fine-tuned for the 16-way RVL-CDIP task.

The model configuration uses a BEiT-style image transformer with:

- 12 transformer layers;
- hidden size 768;
- 12 attention heads;
- 16×16 image patches;
- 224×224 model input; and
- mean pooling before a 16-class classification head.

A key practical consequence is that a potentially very large page image is ultimately compressed into a 224×224 model view.

## 3. Configuration

The default values form the canonical `Run all` path.

In [1]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ""  # @param {type:"string"}
BYOD_LABELS_PATH = ""  # @param {type:"string"}

SAMPLE_SEED = 42  # @param {type:"integer"}
SAMPLE_PER_CLASS = 20  # @param {type:"integer"}

RUN_ROBUSTNESS = True  # @param {type:"boolean"}
RUN_SELECTIVE_CLASSIFICATION = False  # @param {type:"boolean"}

OUTPUT_DIR = "outputs/document_type_classification"

MIN_IMAGE_SIDE = 32
MAX_IMAGE_SIDE = 4096
MAX_PIXELS = 32_000_000
BATCH_SIZE = 16

if SAMPLE_PER_CLASS != 20:
    raise ValueError("The canonical tutorial sample uses exactly 20 pages per class.")
print({
    "sample_seed": SAMPLE_SEED,
    "sample_per_class": SAMPLE_PER_CLASS,
    "run_robustness": RUN_ROBUSTNESS,
    "use_byod": USE_BYOD,
    "output_dir": OUTPUT_DIR,
})

{'sample_seed': 42, 'sample_per_class': 20, 'run_robustness': True, 'use_byod': False, 'output_dir': 'outputs/document_type_classification'}


## 4. Runtime

The notebook uses the current DIMER Python 3.12 / PyTorch runtime family plus the Hugging Face `datasets` library for the compact RVL-CDIP-derived evaluation source.

If the host pre-imports packages that are replaced by the pinned install, the cell fails explicitly instead of continuing with a mixed environment.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

The public PEP 440 version must match each pin; a CUDA local suffix such as `+cu128` does not change its public version. If pre-imported packages were replaced, use **Runtime → Restart session** to keep installed packages, then Run all. This is a restart-assisted path, not verified uninterrupted fresh-runtime execution; that qualification remains pending.


In [2]:
# @title
import importlib.metadata as importlib_metadata
import subprocess
import sys
from packaging.version import Version, InvalidVersion

PINS = {
    "torch": "2.14.0",
    "torchvision": "0.29.0",
    "torchaudio": "2.11.0",
    "transformers": "4.57.6",
    "safetensors": "0.8.0",
    "numpy": "2.1.3",
    "pillow": "11.3.0",
    "huggingface-hub": "0.36.2",
    "datasets": "4.1.1",
}

def installed_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def matches_public_version(observed, expected):
    if observed is None:
        return False
    try:
        return Version(Version(str(observed)).public) == Version(Version(str(expected)).public)
    except InvalidVersion:
        return False

before = {k: installed_version(k) for k in PINS}
needed = [f"{k}=={v}" for k,v in PINS.items() if not matches_public_version(before[k], v)]
if needed:
    print("Installing pinned runtime:", needed)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *needed])

after = {k: installed_version(k) for k in PINS}
bad = {k:(after[k],v) for k,v in PINS.items() if not matches_public_version(after[k], v)}
if bad:
    raise RuntimeError(f"Pinned installation did not converge: {bad}")

stale = []
for module_name, dist_name in [("torch","torch"),("transformers","transformers"),("numpy","numpy")]:
    module = sys.modules.get(module_name)
    if module is not None:
        runtime_v = getattr(module, "__version__", None)
        if runtime_v and not matches_public_version(runtime_v, after[dist_name]):
            stale.append((module_name, runtime_v, after[dist_name]))
if stale:
    raise RuntimeError(
        "This kernel pre-imported packages that were replaced by the pinned install. "
        f"Use Runtime > Restart session (keep installed packages), then Run all again. This restart-assisted run is not uninterrupted fresh-runtime qualification. Stale modules: {stale}"
    )

import numpy as np
import torch
import torchvision
import transformers
import datasets
import huggingface_hub
from PIL import Image, ImageDraw, ImageOps

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32

RUNTIME = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "numpy": np.__version__,
    "pillow": importlib_metadata.version("pillow"),
    "device": DEVICE,
    "cuda_available": torch.cuda.is_available(),
}
if torch.cuda.is_available():
    RUNTIME["gpu_name"] = torch.cuda.get_device_name(0)
    RUNTIME["gpu_total_memory_bytes"] = torch.cuda.get_device_properties(0).total_memory
print(RUNTIME)

Installing pinned runtime: ['torch==2.14.0', 'torchvision==0.29.0', 'transformers==4.57.6', 'huggingface-hub==0.36.2', 'datasets==4.1.1']
{'python': '3.13.15', 'torch': '2.14.0+cu130', 'torchvision': '0.29.0+cu130', 'transformers': '4.57.6', 'datasets': '4.1.1', 'huggingface_hub': '0.36.2', 'numpy': '2.1.3', 'pillow': '11.3.0', 'device': 'cuda:0', 'cuda_available': True, 'gpu_name': 'Tesla T4', 'gpu_total_memory_bytes': 15637086208}


## 5. Model identity and conversion trust boundary

Pinned model identity:

`microsoft/dit-base-finetuned-rvlcdip @ 23f8b03d130fb66bbc9a15df3c75d753e49240eb`

The upstream source checkpoint on this revision is `pytorch_model.bin`, a PyTorch pickle-based serialization. Its expected SHA-256 is pinned in this notebook.

The candidate path is:

1. download only the known model files at the immutable revision;
2. verify the source `.bin` SHA-256 before deserialization;
3. load the state dict with `torch.load(..., weights_only=True)`;
4. instantiate the architecture from the pinned config;
5. load the state dict strictly;
6. save the model state as SafeTensors;
7. reload the converted snapshot;
8. prove logit parity on deterministic probe pages.

After conversion, evaluation uses the SafeTensors model only.

In [3]:
# @title
import hashlib
import json
import shutil
import time
from pathlib import Path
from huggingface_hub import hf_hub_download
from safetensors.torch import save_file

MODEL_ID = "microsoft/dit-base-finetuned-rvlcdip"
MODEL_REVISION = "23f8b03d130fb66bbc9a15df3c75d753e49240eb"
SOURCE_BIN_SHA256 = "1b7a901642c36ec7e32de997683223faf02028624fb2e62a7e7e798a5dd1344e"

SOURCE_DIR = Path("weights/dit-rvlcdip-source")
CONVERTED_DIR = Path("weights/dit-rvlcdip-safetensors")
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
CONVERTED_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for filename in ("config.json", "preprocessor_config.json", "pytorch_model.bin"):
    hf_hub_download(
        repo_id=MODEL_ID,
        filename=filename,
        revision=MODEL_REVISION,
        local_dir=str(SOURCE_DIR),
    )

source_bin = SOURCE_DIR / "pytorch_model.bin"
actual_source_sha = sha256_file(source_bin)
if actual_source_sha != SOURCE_BIN_SHA256:
    raise RuntimeError(
        f"Source checkpoint SHA-256 {actual_source_sha} != pinned {SOURCE_BIN_SHA256}; refusing to load"
    )

config_digest = sha256_file(SOURCE_DIR / "config.json")
processor_digest = sha256_file(SOURCE_DIR / "preprocessor_config.json")

print({
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "source_bin_bytes": source_bin.stat().st_size,
    "source_bin_sha256": actual_source_sha,
    "config_sha256": config_digest,
    "processor_sha256": processor_digest,
})

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/302 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/343M [00:00<?, ?B/s]

{'model_id': 'microsoft/dit-base-finetuned-rvlcdip', 'revision': '23f8b03d130fb66bbc9a15df3c75d753e49240eb', 'source_bin_bytes': 343362393, 'source_bin_sha256': '1b7a901642c36ec7e32de997683223faf02028624fb2e62a7e7e798a5dd1344e', 'config_sha256': '7b7d7bbeff911268dbd0bf735dad4de3614c72e3a9e511a04700068b9f872f6f', 'processor_sha256': 'e04b5163749dcc966092f2dd359d88ed499f1ed5d3d8b1095d9b9d7f54b1c8be'}


## 6. Convert once to SafeTensors and prove parity

The conversion does not alter tensor names, tensor values, dtypes, or shapes.

The source model is instantiated from the verified config and populated using the explicitly `weights_only=True` state dict. The converted model is reconstructed from the new local SafeTensors snapshot.

The probe pages are generated in code so conversion parity does not depend on the evaluation dataset.

In [4]:
# @title
from transformers import AutoConfig, AutoImageProcessor, AutoModelForImageClassification

# Copy load-bearing non-weight assets into the converted snapshot.
shutil.copy2(SOURCE_DIR / "config.json", CONVERTED_DIR / "config.json")
shutil.copy2(SOURCE_DIR / "preprocessor_config.json", CONVERTED_DIR / "preprocessor_config.json")

config = AutoConfig.from_pretrained(str(SOURCE_DIR), local_files_only=True)
expected_labels = [
    "letter","form","email","handwritten","advertisement","scientific report",
    "scientific publication","specification","file folder","news article","budget",
    "invoice","presentation","questionnaire","resume","memo",
]
actual_labels = [config.id2label[i] for i in range(16)]
if actual_labels != expected_labels:
    raise RuntimeError(f"Config class ordering changed: {actual_labels}")

state = torch.load(source_bin, map_location="cpu", weights_only=True)
if not isinstance(state, dict) or not state:
    raise RuntimeError("Source checkpoint did not decode to a non-empty state dict")

source_model = AutoModelForImageClassification.from_config(config)
missing, unexpected = source_model.load_state_dict(state, strict=False)
if missing or unexpected:
    raise RuntimeError(f"Strict state identity failed; missing={missing}, unexpected={unexpected}")
source_model.eval()

converted_path = CONVERTED_DIR / "model.safetensors"
cpu_state = {k: v.detach().cpu().contiguous() for k,v in source_model.state_dict().items()}
save_file(cpu_state, str(converted_path))

converted_sha256 = sha256_file(converted_path)
conversion_manifest = {
    "format": "dimer_derived_model_snapshot",
    "formatVersion": 1,
    "modelKey": "dit-base-finetuned-rvlcdip",
    "modelId": MODEL_ID,
    "revision": MODEL_REVISION,
    "source": {
        "file": "pytorch_model.bin",
        "bytes": source_bin.stat().st_size,
        "sha256": SOURCE_BIN_SHA256,
        "serialization": "PyTorch pickle; loaded only with weights_only=True",
    },
    "derived": {
        "file": "model.safetensors",
        "bytes": converted_path.stat().st_size,
        "sha256": converted_sha256,
        "derivedFromSha256": SOURCE_BIN_SHA256,
    },
    "companions": {
        "config.json": config_digest,
        "preprocessor_config.json": processor_digest,
    },
    "conversion": {
        "torch": torch.__version__,
        "safetensors": importlib_metadata.version("safetensors"),
    },
    "redistributionStatus": "pending-review",
    "hostingStatus": "HOLD",
}
(CONVERTED_DIR / "dimer-base-manifest.json").write_text(
    json.dumps(conversion_manifest, indent=2), encoding="utf-8"
)

processor_probe = AutoImageProcessor.from_pretrained(str(CONVERTED_DIR), local_files_only=True)
converted_model_probe = AutoModelForImageClassification.from_pretrained(
    str(CONVERTED_DIR), local_files_only=True
).eval()

def make_probe(kind):
    image = Image.new("RGB", (900, 1200), "white")
    draw = ImageDraw.Draw(image)
    if kind == "letter":
        for y in range(140, 980, 45):
            draw.line((120,y,760,y), fill="black", width=3)
        draw.rectangle((110,80,420,115), outline="black", width=3)
    else:
        for y in range(100, 1050, 80):
            draw.rectangle((100,y,800,y+35), outline="black", width=2)
        draw.line((450,100,450,1085), fill="black", width=2)
    return image

probe_images = [make_probe("letter"), make_probe("form")]
probe_inputs = processor_probe(images=probe_images, return_tensors="pt")

with torch.inference_mode():
    source_logits = source_model(**probe_inputs).logits
    converted_logits = converted_model_probe(**probe_inputs).logits

max_abs_logit_diff = float((source_logits - converted_logits).abs().max())
if max_abs_logit_diff > 1e-6:
    raise RuntimeError(f"Converted logit parity failed: max abs diff={max_abs_logit_diff}")

if list(source_model.state_dict()) != list(converted_model_probe.state_dict()):
    raise RuntimeError("Converted state-dict key ordering differs from source model")

for key in source_model.state_dict():
    a, b = source_model.state_dict()[key], converted_model_probe.state_dict()[key]
    if a.shape != b.shape or a.dtype != b.dtype or not torch.equal(a, b):
        raise RuntimeError(f"Converted tensor mismatch: {key}")

del state, cpu_state, source_model, converted_model_probe
print({
    "converted_bytes": converted_path.stat().st_size,
    "converted_sha256": converted_sha256,
    "parity_max_abs_logit_diff": max_abs_logit_diff,
    "status": "candidate conversion verified",
})

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.13/dist-packages/transformers/image_processing_base.py:417: UserWarning: The following named arguments are not valid for `BeitImageProcessor.__init__` and were ignored: 'reduce_labels'
  image_processor = cls(**image_processor_dict)


{'converted_bytes': 343305056, 'converted_sha256': 'b9484ea5054bf02ede62493b4e7789a40873a513fb4a94c5aa786869d841d1f7', 'parity_max_abs_logit_diff': 0.0, 'status': 'candidate conversion verified'}


## 7. RVL-CDIP-derived evaluation source

The full RVL-CDIP benchmark contains 400,000 document images and is impractical for a tutorial `Run all`.

For this candidate notebook we use the pinned `hf-tuner/rvl-cdip-document-classification` dataset, which its card states was created from the original `aharley/rvl_cdip` dataset. Its compact test split contains 992 images, balanced at 62 per class.

**Important:** the available dataset card does not prove that those 992 rows are the canonical 40,000-image RVL-CDIP official test split. Therefore this notebook labels the sample:

`balanced RVL-CDIP-derived test subset`

and does not claim official-test benchmark comparability.

A future strict release may replace this acquisition source with a 320-page manifest drawn directly from the official test split without changing the model or evaluation logic.

In [5]:
# @title
from datasets import load_dataset
from huggingface_hub import HfApi

DATASET_ID = "hf-tuner/rvl-cdip-document-classification"
DATASET_EXPECTED_REVISION_PREFIX = "25b73ea"
api = HfApi()
dataset_info = api.repo_info(repo_id=DATASET_ID, repo_type="dataset")
DATASET_REVISION = dataset_info.sha
if not DATASET_REVISION.startswith(DATASET_EXPECTED_REVISION_PREFIX):
    raise RuntimeError(
        f"Dataset revision {DATASET_REVISION} no longer matches expected prefix "
        f"{DATASET_EXPECTED_REVISION_PREFIX}; review upstream changes before running."
    )

dataset_test = load_dataset(
    DATASET_ID,
    split="test",
    revision=DATASET_REVISION,
)

if len(dataset_test) != 992:
    raise RuntimeError(f"Expected 992 rows in the pinned derived test split, got {len(dataset_test)}")

print({
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "test_rows": len(dataset_test),
})

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/463M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/460M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/992 [00:00<?, ? examples/s]

{'dataset_id': 'hf-tuner/rvl-cdip-document-classification', 'dataset_revision': '25b73ea8482e805ef40750464959e910e4579921', 'test_rows': 992}


## 8. Deterministic balanced 320-page tutorial sample

The sample-selection rule uses only the pinned dataset revision, row index, class label, and seed—never model predictions.

For every class:

1. compute `SHA256("42:<row_index>:<class_id>")`;
2. sort rows by that key;
3. retain the first 20.

This produces exactly 320 pages, 20 for each of the 16 classes.

In [6]:
# @title
import io
from collections import Counter, defaultdict

LABELS = [
    "letter","form","email","handwritten","advertisement","scientific report",
    "scientific publication","specification","file folder","news article","budget",
    "invoice","presentation","questionnaire","resume","memo"
]
LABEL_TO_ID = {name:i for i,name in enumerate(LABELS)}

ranked = defaultdict(list)
for row_index in range(len(dataset_test)):
    label_id = int(dataset_test[row_index]["label"])
    key = hashlib.sha256(f"{SAMPLE_SEED}:{row_index}:{label_id}".encode()).hexdigest()
    ranked[label_id].append((key,row_index))

selected_indices = []
for label_id in range(16):
    items = sorted(ranked[label_id])
    if len(items) < SAMPLE_PER_CLASS:
        raise RuntimeError(f"Class {label_id} has only {len(items)} rows")
    selected_indices.extend(idx for _key,idx in items[:SAMPLE_PER_CLASS])

selected_indices = sorted(selected_indices)

def image_bytes_and_rgb(value):
    if isinstance(value, Image.Image):
        image = value.convert("RGB")
        buffer = io.BytesIO()
        image.save(buffer, format="PNG")
        data = buffer.getvalue()
        return data, image
    if isinstance(value, dict) and value.get("bytes") is not None:
        data = value["bytes"]
        image = Image.open(io.BytesIO(data))
        image.load()
        return data, image.convert("RGB")
    raise TypeError(f"Unsupported dataset image type: {type(value).__name__}")

sample_records = []
seen_pixel_digests = set()

for row_index in selected_indices:
    row = dataset_test[row_index]
    raw_bytes, image = image_bytes_and_rgb(row["image"])
    label_id = int(row["label"])
    if label_id not in range(16):
        raise RuntimeError(f"Invalid class id {label_id}")
    if not (MIN_IMAGE_SIDE <= min(image.size) and max(image.size) <= MAX_IMAGE_SIDE):
        raise RuntimeError(f"Row {row_index}: dimensions {image.size} outside tutorial limits")
    if image.width * image.height > MAX_PIXELS:
        raise RuntimeError(f"Row {row_index}: image exceeds {MAX_PIXELS} pixels")
    pixel_digest = hashlib.sha256(
        f"{image.width}x{image.height}:".encode() + image.tobytes()
    ).hexdigest()
    if pixel_digest in seen_pixel_digests:
        raise RuntimeError(f"Duplicate decoded page detected at source row {row_index}")
    seen_pixel_digests.add(pixel_digest)
    sample_records.append({
        "image_id": f"rvl-derived-test-{row_index:04d}",
        "source_row_index": row_index,
        "image": image,
        "source_bytes_sha256": hashlib.sha256(raw_bytes).hexdigest(),
        "pixel_sha256": pixel_digest,
        "width": image.width,
        "height": image.height,
        "label_id": label_id,
        "label": LABELS[label_id],
    })

counts = Counter(r["label_id"] for r in sample_records)
if len(sample_records) != 320 or set(counts.values()) != {20} or len(counts) != 16:
    raise RuntimeError(f"Balanced sample validation failed: {counts}")

sample_digest_payload = "\n".join(
    json.dumps(
        [r["source_row_index"], r["label_id"], r["pixel_sha256"]],
        separators=(",",":"),
    )
    for r in sorted(sample_records, key=lambda x:x["source_row_index"])
)
sample_digest = hashlib.sha256(sample_digest_payload.encode()).hexdigest()

print("Class counts:")
for i,name in enumerate(LABELS):
    print(f"  {i:>2} {name:<24} {counts[i]}")
print({"sample_images":len(sample_records), "sample_digest":sample_digest})

Class counts:
   0 letter                   20
   1 form                     20
   2 email                    20
   3 handwritten              20
   4 advertisement            20
   5 scientific report        20
   6 scientific publication   20
   7 specification            20
   8 file folder              20
   9 news article             20
  10 budget                   20
  11 invoice                  20
  12 presentation             20
  13 questionnaire            20
  14 resume                   20
  15 memo                     20
{'sample_images': 320, 'sample_digest': '4ca6881427c4a95fea66652b75cc8e3b72623f843d33173f49604597ed9fe1d3'}


## 9. Classification baselines

The tutorial sample is exactly balanced.

### Uniform random expectation

`1 / 16 = 6.25%` expected top-1 accuracy.

### Majority baseline

Every class has the same support, so deterministic tie-breaking selects class ID 0: `letter`.

The majority baseline is intentionally trivial. It provides a classification floor, not a useful document system.

> **Before you run it:** predict whether this simple reference will be easy or difficult for the learned model(s) to beat. Record the baseline before interpreting the more complex result.

In [7]:
# @title
true_ids = np.array([r["label_id"] for r in sample_records], dtype=int)
random_accuracy_expectation = 1.0 / len(LABELS)
majority_class_id = min(counts, key=lambda k: (-counts[k], k))
majority_predictions = np.full_like(true_ids, majority_class_id)
majority_accuracy = float(np.mean(majority_predictions == true_ids))

print({
    "random_accuracy_expectation": random_accuracy_expectation,
    "majority_class_id": int(majority_class_id),
    "majority_label": LABELS[majority_class_id],
    "majority_accuracy": majority_accuracy,
})

{'random_accuracy_expectation': 0.0625, 'majority_class_id': 0, 'majority_label': 'letter', 'majority_accuracy': 0.0625}


## 10. Load only the converted model for evaluation

The source pickle checkpoint has already served its only purpose: authenticated-by-digest local conversion.

The evaluation model is reconstructed from:

- `config.json`
- `preprocessor_config.json`
- `model.safetensors`

inside the converted local snapshot.

In [8]:
# @title
import gc

processor = AutoImageProcessor.from_pretrained(str(CONVERTED_DIR), local_files_only=True)

t0 = time.perf_counter()
model = AutoModelForImageClassification.from_pretrained(
    str(CONVERTED_DIR),
    local_files_only=True,
).to(DEVICE).eval()
model_load_seconds = time.perf_counter() - t0

parameter_count = sum(p.numel() for p in model.parameters())
config_labels = [model.config.id2label[i] for i in range(16)]
if config_labels != LABELS:
    raise RuntimeError("Loaded class ordering differs from canonical RVL-CDIP order")

print({
    "model_load_seconds": model_load_seconds,
    "parameter_count": parameter_count,
    "class_count": len(config_labels),
    "device": DEVICE,
})

{'model_load_seconds': 0.5582987089999278, 'parameter_count': 85820176, 'class_count': 16, 'device': 'cuda:0'}


## 11. Whole-page inference

For every page, the processor creates the model's 224×224 view and the classifier returns 16 logits.

The notebook exports the complete 16-class softmax score vector and uses `argmax` as the decision rule.

Softmax scores are **normalized model scores over this fixed class set**, not calibrated probabilities of real-world correctness.

**Predict before running:** which two document types might share page-layout cues and be confused? Write a pair and a visual reason. After inference, compare your prediction with the confusion matrix and deterministic error gallery, not just overall accuracy.


In [9]:
# @title
def batched_predict(records, batch_size=BATCH_SIZE):
    all_logits = []
    timings = []
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # Warm-up.
    warm = processor(images=[records[0]["image"]], return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        _ = model(**warm).logits
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    for start in range(0, len(records), batch_size):
        batch_records = records[start:start+batch_size]
        inputs = processor(
            images=[r["image"] for r in batch_records],
            return_tensors="pt",
        ).to(DEVICE)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.inference_mode():
            logits = model(**inputs).logits
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        timings.append(time.perf_counter() - t0)
        if logits.shape != (len(batch_records), 16):
            raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")
        if not torch.isfinite(logits).all():
            raise RuntimeError("Model produced non-finite logits")
        all_logits.append(logits.float().cpu())

    logits = torch.cat(all_logits, dim=0).numpy()
    scores = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    if not np.allclose(scores.sum(axis=1), 1.0, atol=1e-5):
        raise RuntimeError("Softmax score rows do not sum approximately to 1")
    peak_gpu = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
    return logits, scores, timings, peak_gpu

logits, scores, inference_batch_times, peak_gpu_memory = batched_predict(sample_records)
pred_ids = scores.argmax(axis=1)
top3_ids = np.argsort(-scores, axis=1)[:,:3]

print({
    "pages": len(sample_records),
    "batches": len(inference_batch_times),
    "total_model_forward_seconds": float(sum(inference_batch_times)),
    "mean_batch_seconds": float(np.mean(inference_batch_times)),
    "peak_gpu_memory_bytes": peak_gpu_memory,
})

{'pages': 320, 'batches': 20, 'total_model_forward_seconds': 3.0464076450004995, 'mean_batch_seconds': 0.15232038225002498, 'peak_gpu_memory_bytes': 498254848}


## 12. Principal metrics

We report three complementary classification measures:

- **Accuracy** — top-1 correctness.
- **Macro F1** — equal-weight F1 across all 16 classes.
- **Top-3 accuracy** — whether the gold class appears among the three highest scores.

The sample metrics are tutorial evidence on the pinned RVL-CDIP-derived subset, not reproduction of the original 40k official benchmark.

> **What to notice.** Compare the learned-model result with the baseline/reference first, then use the secondary diagnostics to explain the behavior. Do not infer a universal model ranking from one tutorial sample and configuration.

**Checkpoint:** could a high overall accuracy hide a weak class?

<details><summary>Sample interpretation</summary>
Yes. Inspect per-class recall and macro-F1 alongside the overall score. Here the sample is balanced, but classes can still have very different recall. A small tutorial sample cannot establish general document performance.
</details>


In [10]:
# @title
def confusion_matrix_np(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes,n_classes), dtype=int)
    for t,p in zip(y_true,y_pred,strict=True):
        cm[int(t),int(p)] += 1
    return cm

def per_class_metrics_from_cm(cm):
    rows = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fp = cm[:,i].sum() - tp
        fn = cm[i,:].sum() - tp
        support = cm[i,:].sum()
        precision = tp/(tp+fp) if tp+fp else 0.0
        recall = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
        rows.append({
            "class_id": i,
            "class_label": LABELS[i],
            "support": int(support),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
        })
    return rows

cm = confusion_matrix_np(true_ids, pred_ids, 16)
class_metrics = per_class_metrics_from_cm(cm)
accuracy = float(np.mean(pred_ids == true_ids))
macro_f1 = float(np.mean([r["f1"] for r in class_metrics]))
top3_accuracy = float(np.mean([
    int(true_ids[i]) in set(int(x) for x in top3_ids[i])
    for i in range(len(true_ids))
]))

for i,row in enumerate(class_metrics):
    row["top3_recall"] = float(np.mean([
        i in set(int(x) for x in top3_ids[j])
        for j in range(len(true_ids))
        if true_ids[j] == i
    ]))

metrics = {
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "top3_accuracy": top3_accuracy,
    "random_accuracy_expectation": random_accuracy_expectation,
    "majority_accuracy": majority_accuracy,
}
print(metrics)

{'accuracy': 0.953125, 'macro_f1': 0.9530115891276032, 'top3_accuracy': 0.99375, 'random_accuracy_expectation': 0.0625, 'majority_accuracy': 0.0625}


## 13. Confusion matrix and prediction

Before reading the matrix, consider which page types you expect to be visually confusable.

Possible hypotheses include:

- scientific report ↔ scientific publication;
- letter ↔ memo;
- form ↔ questionnaire;
- budget ↔ invoice.

These are only hypotheses. The next cell reports the actual measured confusions.

In [11]:
# @title
print("Raw confusion matrix (rows=gold, columns=prediction):")
header = "      " + " ".join(f"{i:>3}" for i in range(16))
print(header)
for i,row in enumerate(cm):
    print(f"{i:>3} {LABELS[i][:14]:<14} " + " ".join(f"{int(v):>3}" for v in row))

off_diagonal = []
for i in range(16):
    for j in range(16):
        if i != j and cm[i,j] > 0:
            off_diagonal.append((int(cm[i,j]), i, j))
off_diagonal.sort(reverse=True)

print("\nStrongest measured confusion pairs:")
for count,i,j in off_diagonal[:10]:
    print(f"  {LABELS[i]} -> {LABELS[j]}: {count}")

Raw confusion matrix (rows=gold, columns=prediction):
        0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15
  0 letter          18   1   0   0   0   0   0   0   0   0   0   0   1   0   0   0
  1 form             0  18   0   0   0   1   0   0   0   0   1   0   0   0   0   0
  2 email            0   0  20   0   0   0   0   0   0   0   0   0   0   0   0   0
  3 handwritten      0   0   0  18   0   0   0   0   0   0   0   0   0   2   0   0
  4 advertisement    0   0   0   0  20   0   0   0   0   0   0   0   0   0   0   0
  5 scientific rep   0   0   0   0   1  17   0   0   0   0   0   0   2   0   0   0
  6 scientific pub   0   0   0   0   0   0  19   0   0   1   0   0   0   0   0   0
  7 specification    0   0   0   0   0   0   0  20   0   0   0   0   0   0   0   0
  8 file folder      0   0   0   0   0   0   0   0  20   0   0   0   0   0   0   0
  9 news article     0   0   0   0   1   0   0   0   0  19   0   0   0   0   0   0
 10 budget           0   0   0   0   0   0   0

## 14. Error analysis

For each page we calculate:

- top-1 score;
- top-2 score;
- top-1/top-2 margin;
- score entropy;
- correctness.

We then surface:

- highest-score incorrect predictions;
- smallest-margin predictions;
- correct pages with the smallest margins.

A high model score is not proof of correctness.

In [12]:
# @title
eps = 1e-12
prediction_rows = []

for i,record in enumerate(sample_records):
    order = np.argsort(-scores[i])
    top1, top2, top3 = [int(x) for x in order[:3]]
    entropy = float(-np.sum(scores[i] * np.log(np.maximum(scores[i], eps))))
    row = {
        "image_id": record["image_id"],
        "source_row_index": record["source_row_index"],
        "true_label": record["label"],
        "true_class_id": record["label_id"],
        "predicted_class_id": top1,
        "predicted_label": LABELS[top1],
        "correct": bool(top1 == record["label_id"]),
        "top1_score": float(scores[i,top1]),
        "top2_label": LABELS[top2],
        "top2_score": float(scores[i,top2]),
        "top3_label": LABELS[top3],
        "top3_score": float(scores[i,top3]),
        "margin": float(scores[i,top1] - scores[i,top2]),
        "entropy": entropy,
    }
    prediction_rows.append(row)

high_conf_wrong = sorted(
    [r for r in prediction_rows if not r["correct"]],
    key=lambda r: -r["top1_score"],
)[:8]
low_margin = sorted(prediction_rows, key=lambda r:r["margin"])[:8]
correct_uncertain = sorted(
    [r for r in prediction_rows if r["correct"]],
    key=lambda r:r["margin"],
)[:8]

print("High-confidence incorrect:")
for r in high_conf_wrong:
    print(f"  {r['image_id']}: gold={r['true_label']} pred={r['predicted_label']} score={r['top1_score']:.3f} margin={r['margin']:.3f}")

print("\nLowest margins:")
for r in low_margin:
    print(f"  {r['image_id']}: gold={r['true_label']} pred={r['predicted_label']} margin={r['margin']:.4f}")

High-confidence incorrect:
  rvl-derived-test-0123: gold=presentation pred=scientific report score=0.896 margin=0.883
  rvl-derived-test-0960: gold=news article pred=advertisement score=0.888 margin=0.856
  rvl-derived-test-0908: gold=letter pred=presentation score=0.839 margin=0.808
  rvl-derived-test-0436: gold=scientific report pred=presentation score=0.833 margin=0.787
  rvl-derived-test-0319: gold=handwritten pred=questionnaire score=0.815 margin=0.775
  rvl-derived-test-0628: gold=form pred=budget score=0.797 margin=0.682
  rvl-derived-test-0861: gold=presentation pred=scientific publication score=0.765 margin=0.627
  rvl-derived-test-0139: gold=handwritten pred=questionnaire score=0.761 margin=0.609

Lowest margins:
  rvl-derived-test-0244: gold=budget pred=memo margin=0.0090
  rvl-derived-test-0504: gold=scientific report pred=presentation margin=0.0437
  rvl-derived-test-0411: gold=scientific report pred=scientific report margin=0.1331
  rvl-derived-test-0852: gold=presentatio

## 15. Deterministic robustness probe

The robustness probe uses one page per class, selected only by the original sample-selection ranking key.

Each probe keeps the same ground-truth class while the visual evidence changes.

Variants:

- original;
- rotate 90° clockwise;
- rotate 180°;
- low resolution (downsample to 112×112, then re-upsample);
- center crop retaining 80% of width and height.

These transformations measure stability. They are not production preprocessing recommendations.

In [13]:
# @title
# Reconstruct the deterministic ranking key used during selection and pick the first sample per class.
def selection_key(record):
    idx = record["source_row_index"]
    label_id = record["label_id"]
    return hashlib.sha256(f"{SAMPLE_SEED}:{idx}:{label_id}".encode()).hexdigest()

robustness_base = []
for label_id in range(16):
    candidates = [r for r in sample_records if r["label_id"] == label_id]
    robustness_base.append(min(candidates, key=selection_key))

print("Robustness probe:")
for r in robustness_base:
    print(f"  {r['label_id']:>2} {r['label']:<24} {r['image_id']}")

Robustness probe:
   0 letter                   rvl-derived-test-0587
   1 form                     rvl-derived-test-0775
   2 email                    rvl-derived-test-0270
   3 handwritten              rvl-derived-test-0319
   4 advertisement            rvl-derived-test-0141
   5 scientific report        rvl-derived-test-0928
   6 scientific publication   rvl-derived-test-0521
   7 specification            rvl-derived-test-0063
   8 file folder              rvl-derived-test-0865
   9 news article             rvl-derived-test-0901
  10 budget                   rvl-derived-test-0298
  11 invoice                  rvl-derived-test-0465
  12 presentation             rvl-derived-test-0861
  13 questionnaire            rvl-derived-test-0413
  14 resume                   rvl-derived-test-0213
  15 memo                     rvl-derived-test-0272


## 16. Rotation, resolution and crop experiments

The same classifier and processor are used for every variant. Only the input page image changes.

**Predict before running:** will rotating the same page by 90 degrees hurt more than downsampling it? Give a layout-based reason. `RUN_ROBUSTNESS=False` skips all transformed-model calls and exports an empty robustness table. With the default enabled, compare each variant against the original for exactly the same 16 pages.


In [14]:
# @title
def transform_variant(image, variant):
    image = image.convert("RGB")
    if variant == "original":
        return image
    if variant == "rotate90":
        return image.rotate(-90, expand=True, fillcolor="white")
    if variant == "rotate180":
        return image.rotate(180, expand=True, fillcolor="white")
    if variant == "low_resolution":
        low = image.resize((112,112), Image.Resampling.BILINEAR)
        return low.resize(image.size, Image.Resampling.BILINEAR)
    if variant == "center_crop":
        w,h = image.size
        cw,ch = int(w*0.8), int(h*0.8)
        x0,y0 = (w-cw)//2, (h-ch)//2
        return image.crop((x0,y0,x0+cw,y0+ch))
    raise ValueError(variant)

ROBUSTNESS_VARIANTS = ["original","rotate90","rotate180","low_resolution","center_crop"]

robustness_rows = []
robustness_summary = {}
robust_records = []
if RUN_ROBUSTNESS:
    robust_records = []
    for base in robustness_base:
        for variant in ROBUSTNESS_VARIANTS:
            robust_records.append({
                **{k:v for k,v in base.items() if k != "image"},
                "variant": variant,
                "image": transform_variant(base["image"], variant),
            })

    robust_logits, robust_scores, _robust_times, _ = batched_predict(robust_records, batch_size=16)
    robust_pred_ids = robust_scores.argmax(axis=1)

    robustness_rows = []
    original_prediction_by_id = {}

    for i,r in enumerate(robust_records):
        order = np.argsort(-robust_scores[i])
        top1,top2 = int(order[0]),int(order[1])
        if r["variant"] == "original":
            original_prediction_by_id[r["image_id"]] = top1
        robustness_rows.append({
            "image_id": r["image_id"],
            "gold_label": r["label"],
            "gold_class_id": r["label_id"],
            "variant": r["variant"],
            "predicted_label": LABELS[top1],
            "predicted_class_id": top1,
            "correct": bool(top1 == r["label_id"]),
            "top1_score": float(robust_scores[i,top1]),
            "gold_score": float(robust_scores[i,r["label_id"]]),
            "margin": float(robust_scores[i,top1] - robust_scores[i,top2]),
        })

    for row in robustness_rows:
        row["same_as_original_prediction"] = (
            row["predicted_class_id"] == original_prediction_by_id[row["image_id"]]
        )

    robustness_summary = {}
    for variant in ROBUSTNESS_VARIANTS:
        rows = [r for r in robustness_rows if r["variant"] == variant]
        robustness_summary[variant] = {
            "accuracy": float(np.mean([r["correct"] for r in rows])),
            "stability_vs_original": float(np.mean([r["same_as_original_prediction"] for r in rows])),
            "mean_gold_score": float(np.mean([r["gold_score"] for r in rows])),
            "mean_margin": float(np.mean([r["margin"] for r in rows])),
        }

    print(f"{'Variant':<16} {'Accuracy':>10} {'Stability':>10} {'Gold score':>11} {'Margin':>9}")
    for variant,summary in robustness_summary.items():
        print(
            f"{variant:<16} {summary['accuracy']:>10.3f} {summary['stability_vs_original']:>10.3f} "
            f"{summary['mean_gold_score']:>11.3f} {summary['mean_margin']:>9.3f}"
        )
else:
    print("Robustness disabled; exports contain an empty robustness table and summary.")


Variant            Accuracy  Stability  Gold score    Margin
original              0.812      1.000       0.745     0.846
rotate90              0.500      0.625       0.438     0.556
rotate180             0.688      0.875       0.625     0.843
low_resolution        0.750      0.938       0.723     0.766
center_crop           0.812      1.000       0.751     0.838


## 17. Confidence, margins and optional selective classification

The notebook exposes scores, margins and entropy as diagnostics, but does not define a universal rejection threshold.

An optional experiment can measure the trade-off between **coverage** and **accuracy among retained pages** as the top-1/top-2 margin threshold increases.

This is exploratory calibration behavior on the tutorial sample, not a deployment threshold recommendation.

In [15]:
# @title
selective_rows = []
if RUN_SELECTIVE_CLASSIFICATION:
    for threshold in (0.0,0.05,0.10,0.20,0.30,0.50):
        kept = [r for r in prediction_rows if r["margin"] >= threshold]
        selective_rows.append({
            "margin_threshold": threshold,
            "coverage": len(kept)/len(prediction_rows),
            "accuracy_if_retained": (
                float(np.mean([r["correct"] for r in kept])) if kept else None
            ),
            "n_retained": len(kept),
        })
    for r in selective_rows:
        print(r)
else:
    print("Selective-classification experiment disabled on the canonical path.")

Selective-classification experiment disabled on the canonical path.


## 18. Visual galleries

The first gallery contains one deterministic page per class.

The second contains the highest-confidence incorrect predictions available in the sample.

Each thumbnail is annotated with the gold class, predicted class and top-1 score.

In [16]:
# @title
import math

def make_gallery(records_with_rows, columns=4, thumb_size=(260,340), title_height=48):
    tile_w,tile_h = thumb_size[0], thumb_size[1] + title_height
    rows_n = math.ceil(len(records_with_rows)/columns)
    canvas = Image.new("RGB", (columns*tile_w, rows_n*tile_h), "white")
    draw = ImageDraw.Draw(canvas)
    for idx,(record,pred_row) in enumerate(records_with_rows):
        col,row = idx%columns, idx//columns
        img = record["image"].convert("RGB").copy()
        img.thumbnail(thumb_size)
        tile = Image.new("RGB", thumb_size, "white")
        tile.paste(img, ((thumb_size[0]-img.width)//2, (thumb_size[1]-img.height)//2))
        x,y = col*tile_w, row*tile_h
        canvas.paste(tile, (x,y+title_height))
        text1 = f"G: {pred_row['true_label'][:20]}"
        text2 = f"P: {pred_row['predicted_label'][:18]} {pred_row['top1_score']:.2f}"
        draw.text((x+4,y+4), text1, fill="black")
        draw.text((x+4,y+24), text2, fill="black")
    return canvas

record_by_id = {r["image_id"]:r for r in sample_records}
pred_by_id = {r["image_id"]:r for r in prediction_rows}

class_gallery_items = [
    (robustness_base[i], pred_by_id[robustness_base[i]["image_id"]])
    for i in range(16)
]

error_gallery_items = [
    (record_by_id[r["image_id"]], r)
    for r in high_conf_wrong
]

out_dir = Path(OUTPUT_DIR)
gallery_dir = out_dir / "galleries"
gallery_dir.mkdir(parents=True, exist_ok=True)

class_gallery = make_gallery(class_gallery_items, columns=4)
class_gallery_path = gallery_dir / "class_gallery.png"
class_gallery.save(class_gallery_path)

if error_gallery_items:
    error_gallery = make_gallery(error_gallery_items, columns=4)
    error_gallery_path = gallery_dir / "high_confidence_errors.png"
    error_gallery.save(error_gallery_path)
else:
    error_gallery_path = None

print(class_gallery_path)
print(error_gallery_path)

outputs/document_type_classification/galleries/class_gallery.png
outputs/document_type_classification/galleries/high_confidence_errors.png


## 19. Machine-readable outputs and provenance

The notebook exports:

- `predictions.csv`
- `class_scores.csv`
- `class_metrics.csv`
- `confusion_matrix.csv`
- `robustness.csv`
- `input_manifest.json`
- `metrics.json`
- `provenance.json`
- gallery PNGs

The provenance distinguishes the upstream pickle source asset from the derived SafeTensors file used for evaluation.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [17]:
# @title
import csv
from datetime import datetime, timezone

out_dir.mkdir(parents=True, exist_ok=True)

def write_csv(path, rows, fieldnames):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for row in rows:
            w.writerow({k:row.get(k) for k in fieldnames})

write_csv(
    out_dir/"predictions.csv",
    prediction_rows,
    [
        "image_id","source_row_index","true_label","true_class_id","predicted_class_id",
        "predicted_label","correct","top1_score","top2_label","top2_score","top3_label",
        "top3_score","margin","entropy"
    ],
)

class_score_rows = []
for i,record in enumerate(sample_records):
    order = np.argsort(-scores[i])
    rank_by_class = {int(cls):rank+1 for rank,cls in enumerate(order)}
    for class_id,label in enumerate(LABELS):
        class_score_rows.append({
            "image_id": record["image_id"],
            "class_id": class_id,
            "class_label": label,
            "score": float(scores[i,class_id]),
            "rank": rank_by_class[class_id],
        })

if len(class_score_rows) != len(sample_records)*16:
    raise RuntimeError("class_scores export does not contain exactly 16 rows per page")

write_csv(
    out_dir/"class_scores.csv",
    class_score_rows,
    ["image_id","class_id","class_label","score","rank"],
)

write_csv(
    out_dir/"class_metrics.csv",
    class_metrics,
    ["class_id","class_label","support","precision","recall","f1","top3_recall"],
)

cm_rows = []
for gold_id in range(16):
    row = {"gold_class_id":gold_id,"gold_label":LABELS[gold_id]}
    for pred_id in range(16):
        row[f"pred_{pred_id}"] = int(cm[gold_id,pred_id])
    cm_rows.append(row)
write_csv(
    out_dir/"confusion_matrix.csv",
    cm_rows,
    ["gold_class_id","gold_label"] + [f"pred_{i}" for i in range(16)],
)

write_csv(
    out_dir/"robustness.csv",
    robustness_rows,
    [
        "image_id","gold_label","gold_class_id","variant","predicted_label","predicted_class_id",
        "correct","top1_score","gold_score","margin","same_as_original_prediction"
    ],
)

input_manifest = {
    "schema": {
        "unit": "one document page image",
        "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
        "max_pixels": MAX_PIXELS,
        "classes": LABELS,
        "decision_rule": "argmax over 16 logits",
    },
    "sample": {
        "images": len(sample_records),
        "class_counts": {LABELS[i]:counts[i] for i in range(16)},
        "min_width": min(r["width"] for r in sample_records),
        "max_width": max(r["width"] for r in sample_records),
        "min_height": min(r["height"] for r in sample_records),
        "max_height": max(r["height"] for r in sample_records),
    },
    "verdict": "accepted",
    "findings": [
        "rejection probe: a synthetic 16x16 image would fail MIN_IMAGE_SIDE=32"
    ],
}
(out_dir/"input_manifest.json").write_text(json.dumps(input_manifest, indent=2), encoding="utf-8")

metrics_export = {
    **metrics,
    "per_class": class_metrics,
    "confusion_matrix": cm.tolist(),
    "strongest_confusions": [
        {"count":count,"gold":LABELS[i],"predicted":LABELS[j]}
        for count,i,j in off_diagonal[:10]
    ],
    "robustness": robustness_summary,
    "selective_classification": selective_rows,
    "timing": {
        "model_load_seconds": model_load_seconds,
        "batch_forward_seconds": inference_batch_times,
        "total_forward_seconds": float(sum(inference_batch_times)),
    },
}
(out_dir/"metrics.json").write_text(json.dumps(metrics_export, indent=2), encoding="utf-8")

provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_spec": "2.1",
    "profile": "TASK-INFERENCE",
    "pedagogical_mode": "WORKSHOP",
    "standalone": True,
    "model": {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,
        "class_order": LABELS,
        "parameter_count": parameter_count,
        "source_asset": {
            "file": "pytorch_model.bin",
            "bytes": source_bin.stat().st_size,
            "sha256": SOURCE_BIN_SHA256,
            "serialization": "PyTorch pickle; verified then loaded with weights_only=True",
        },
        "derived_asset": {
            "file": "model.safetensors",
            "bytes": converted_path.stat().st_size,
            "sha256": converted_sha256,
            "derived_from_sha256": SOURCE_BIN_SHA256,
        },
        "conversion_parity_max_abs_logit_diff": max_abs_logit_diff,
        "config_sha256": config_digest,
        "preprocessor_sha256": processor_digest,
        "redistribution_status": "pending-review",
        "hosting_status": "HOLD",
    },
    "dataset": {
        "id": DATASET_ID,
        "revision": DATASET_REVISION,
        "source_description": "balanced RVL-CDIP-derived dataset; test split contains 992 images / 62 per class",
        "sample_kind": "balanced RVL-CDIP-derived test tutorial subset",
        "sample_seed": SAMPLE_SEED,
        "sample_per_class": SAMPLE_PER_CLASS,
        "sample_images": len(sample_records),
        "sample_digest": sample_digest,
        "sample_manifest": [
            {
                "image_id":r["image_id"],
                "source_row_index":r["source_row_index"],
                "class_id":r["label_id"],
                "class_label":r["label"],
                "width":r["width"],
                "height":r["height"],
                "source_bytes_sha256":r["source_bytes_sha256"],
                "pixel_sha256":r["pixel_sha256"],
            }
            for r in sample_records
        ],
    },
    "runtime": RUNTIME,
    "device": DEVICE,
}
(out_dir/"provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")

print("Exports:")
for p in sorted(out_dir.iterdir()):
    print(" ", p)

Exports:
  outputs/document_type_classification/class_metrics.csv
  outputs/document_type_classification/class_scores.csv
  outputs/document_type_classification/confusion_matrix.csv
  outputs/document_type_classification/galleries
  outputs/document_type_classification/input_manifest.json
  outputs/document_type_classification/metrics.json
  outputs/document_type_classification/predictions.csv
  outputs/document_type_classification/provenance.json
  outputs/document_type_classification/robustness.csv


## 20. Bring Your Own Document Pages (optional)

BYOD is disabled on the canonical path.

Accepted input:

- one image file;
- a directory of page images; or
- a ZIP containing page images.

Optional `labels.csv`:

`filename,label`

Labels must be one of the fixed 16 RVL-CDIP classes.

If labels are absent, the notebook reports predictions only and the evaluation verdict is `not-measurable`.

### Privacy

Document images can contain names, addresses, signatures, financial information, proprietary records, and personal correspondence. Do not upload restricted, confidential, personal, or regulated documents to a hosted notebook environment unless you are authorized to process them there.

**Input limits:** 1–200 page images, sides 32–4096 pixels and at most 32,000,000 pixels per page; ZIP expanded size at most 512 MiB. ZIP basenames must be unique even across folders; each attempt extracts to a new directory so prior pages cannot leak into the input. PDFs must first be rendered to page images.

When a labels CSV is supplied, it must contain exactly one valid label for every selected filename: duplicate, missing or extra names are rejected before inference. Without the CSV, all pages are prediction-only. Image IDs include the filename extension. The branch writes separate `byod_results.json` and `byod_predictions.csv`, with per-image and optional ZIP/labels digests, model revision and runtime. Supplied pages remain in this runtime; no DIMER service is used.


In [18]:
# @title
import zipfile
import tempfile
import io

BYOD_EXTENSIONS = {".png",".jpg",".jpeg",".webp",".tif",".tiff"}

def safe_extract_images(zip_path, dest):
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        expanded = 0
        members = []
        seen = set()
        for info in zf.infolist():
            name = info.filename.replace("\\", "/")
            parts = [part for part in name.split("/") if part]
            if info.is_dir():
                continue
            if name.startswith("/") or ".." in parts or any(":" in part for part in parts):
                raise ValueError(f"Unsafe ZIP member: {name}")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == 0o120000:
                raise ValueError(f"Symlink ZIP member refused: {name}")
            expanded += info.file_size
            if expanded > 512*1024*1024:
                raise ValueError("BYOD ZIP exceeds 512 MiB expanded-size ceiling")
            if Path(name).suffix.lower() in BYOD_EXTENSIONS:
                basename = Path(name).name
                if basename.casefold() in seen:
                    raise ValueError(f"Duplicate image basename in ZIP: {basename}")
                seen.add(basename.casefold())
                members.append((info, basename))
        if not 1 <= len(members) <= 200:
            raise ValueError("BYOD ZIP must contain 1..200 page images")
        # Unique directory per attempt; never mix prior runs or overwrite their files.
        extraction = Path(tempfile.mkdtemp(prefix="pages-", dir=dest))
        for info, basename in members:
            (extraction / basename).write_bytes(zf.read(info))
    return extraction

def validate_byod_image(path):
    image = Image.open(path)
    image.load()
    image = image.convert("RGB")
    if not (MIN_IMAGE_SIDE <= min(image.size) and max(image.size) <= MAX_IMAGE_SIDE):
        raise ValueError(f"{path.name}: side outside {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE}")
    if image.width*image.height > MAX_PIXELS:
        raise ValueError(f"{path.name}: exceeds {MAX_PIXELS} pixels")
    return image

byod_result = None
if USE_BYOD:
    source = Path(BYOD_PATH)
    if source.is_file() and source.suffix.lower() == ".zip":
        image_root = safe_extract_images(source, Path("byod_document_pages"))
        image_paths = sorted(p for p in image_root.iterdir() if p.is_file() and p.suffix.lower() in BYOD_EXTENSIONS)
    elif source.is_dir():
        image_paths = sorted(p for p in source.iterdir() if p.is_file() and p.suffix.lower() in BYOD_EXTENSIONS)
    elif source.is_file() and source.suffix.lower() in BYOD_EXTENSIONS:
        image_paths = [source]
    else:
        raise FileNotFoundError("BYOD_PATH must be an image, image directory, or ZIP")

    if not 1 <= len(image_paths) <= 200:
        raise ValueError("BYOD accepts 1..200 page images")

    image_names = [p.name for p in image_paths]
    if len({name.casefold() for name in image_names}) != len(image_names):
        raise ValueError("BYOD image filenames must be unique (case-insensitive)")
    image_digests = {p.name: sha256_file(p) for p in image_paths}
    labels_digest = None
    labels_by_name = {}
    if BYOD_LABELS_PATH:
        labels_bytes = Path(BYOD_LABELS_PATH).read_bytes()
        labels_digest = hashlib.sha256(labels_bytes).hexdigest()
        reader = csv.DictReader(io.StringIO(labels_bytes.decode("utf-8-sig")))
        rows = list(reader)
        if not rows or not {"filename", "label"}.issubset(reader.fieldnames or []):
            raise ValueError("labels.csv must contain filename,label and one row per image")
        seen_label_names = set()
        for row in rows:
            filename = (row.get("filename") or "").strip()
            label = (row.get("label") or "").strip().lower()
            if not filename or filename.casefold() in seen_label_names:
                raise ValueError(f"Duplicate or empty label filename: {filename!r}")
            seen_label_names.add(filename.casefold())
            if label not in LABEL_TO_ID:
                raise ValueError(f"Unknown RVL-CDIP label {label!r}")
            labels_by_name[filename] = label
        missing = sorted(set(image_names) - set(labels_by_name))
        extra = sorted(set(labels_by_name) - set(image_names))
        if missing or extra:
            raise ValueError(f"Label filenames must match images exactly; missing={missing}, extra={extra}")

    byod_records = []
    for p in image_paths:
        image = validate_byod_image(p)
        label = labels_by_name.get(p.name)
        byod_records.append({
            "image_id": p.name,
            "image": image,
            "label": label,
            "label_id": LABEL_TO_ID[label] if label is not None else None,
        })

    _l, byod_scores, _t, _m = batched_predict(byod_records, batch_size=min(BATCH_SIZE,len(byod_records)))
    byod_pred = byod_scores.argmax(axis=1)
    byod_rows = []
    for i,r in enumerate(byod_records):
        order = np.argsort(-byod_scores[i])
        byod_rows.append({
            "image_id":r["image_id"],
            "true_label":r["label"],
            "predicted_label":LABELS[int(order[0])],
            "top1_score":float(byod_scores[i,order[0]]),
            "top2_label":LABELS[int(order[1])],
            "top2_score":float(byod_scores[i,order[1]]),
        })
    labelled = all(r["label_id"] is not None for r in byod_records)
    byod_result = {
        "images":len(byod_records),
        "evaluation_verdict":"measured" if labelled else "not-measurable",
        "predictions":byod_rows,
    }
    if labelled:
        y = np.array([r["label_id"] for r in byod_records],dtype=int)
        p = byod_pred
        btop3 = np.argsort(-byod_scores,axis=1)[:,:3]
        bcm = confusion_matrix_np(y,p,16)
        bmetrics = per_class_metrics_from_cm(bcm)
        byod_result["accuracy"] = float(np.mean(y==p))
        byod_result["macro_f1"] = float(np.mean([r["f1"] for r in bmetrics]))
        byod_result["top3_accuracy"] = float(np.mean([y[i] in btop3[i] for i in range(len(y))]))
    byod_result["provenance"] = {
        "image_sha256": image_digests,
        "source_zip_sha256": sha256_file(source) if source.is_file() and source.suffix.lower() == ".zip" else None,
        "labels_sha256": labels_digest,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "converted_safetensors_sha256": converted_sha256,
        "class_order": LABELS,
        "runtime": RUNTIME,
    }
    byod_out_dir = Path(OUTPUT_DIR)
    byod_out_dir.mkdir(parents=True, exist_ok=True)
    (byod_out_dir / "byod_results.json").write_text(json.dumps(byod_result, indent=2), encoding="utf-8")
    write_csv(byod_out_dir / "byod_predictions.csv", byod_rows,
              ["image_id", "true_label", "predicted_label", "top1_score", "top2_label", "top2_score"])
    print({k:v for k,v in byod_result.items() if k!="predictions"})
else:
    print("BYOD disabled on the canonical Run all path.")

BYOD disabled on the canonical Run all path.


## 21. Limitations and transfer

### Fixed vocabulary

The checkpoint can output only its 16 RVL-CDIP classes. Renaming the labels does not create new concepts.

A custom organizational taxonomy such as `purchase order`, `passport`, `tax return`, or `government memorandum` requires supervised adaptation with a new classification head.

### Single-page boundary

Each input image is classified independently. Multi-page document aggregation is outside this notebook.

### No OCR requirement does not mean text is irrelevant

The model sees raster pixels. Text appearance, typography and page layout can all contribute visually, but the notebook does not extract text or expose OCR tokens.

### Evaluation-source limitation

This candidate implementation uses a compact dataset explicitly derived from RVL-CDIP. Its card does not establish equivalence to the canonical 40k official test split, so these results must not be compared directly with published official-test numbers.

### Asset-hosting limitation

The candidate notebook demonstrates safe local conversion from the pinned upstream pickle source. A release-grade DIMER profile should instead distribute the reviewed converted SafeTensors snapshot after redistribution clearance.

## 22. Terminal summary

The final cell verifies the required outputs and prints only measurements from the current execution.

In [19]:
# @title
required_outputs = [
    out_dir/"predictions.csv",
    out_dir/"class_scores.csv",
    out_dir/"class_metrics.csv",
    out_dir/"confusion_matrix.csv",
    out_dir/"robustness.csv",
    out_dir/"input_manifest.json",
    out_dir/"metrics.json",
    out_dir/"provenance.json",
]
if USE_BYOD:
    required_outputs.extend([out_dir/"byod_results.json", out_dir/"byod_predictions.csv"])
missing = [str(p) for p in required_outputs if not p.is_file()]
if missing:
    raise RuntimeError(f"Required outputs missing: {missing}")

print("DIMER Document-Type Classification Notebook")
print("-"*45)
print(f"Sample: {len(sample_records)} balanced RVL-CDIP-derived pages")
print(f"Classes: {len(LABELS)}")
print()
print(f"Random expectation: {random_accuracy_expectation:.4f}")
print(f"Majority baseline:  {majority_accuracy:.4f}")
print(f"Accuracy:           {accuracy:.4f}")
print(f"Macro F1:           {macro_f1:.4f}")
print(f"Top-3 accuracy:     {top3_accuracy:.4f}")
print()
print(f"Source checkpoint SHA-256: {SOURCE_BIN_SHA256}")
print(f"Converted SafeTensors SHA-256: {converted_sha256}")
print(f"Conversion parity max abs logit diff: {max_abs_logit_diff:.3e}")
print()
print("Robustness:")
for variant,summary in robustness_summary.items():
    print(f"  {variant:<16} accuracy={summary['accuracy']:.3f} stability={summary['stability_vs_original']:.3f}")
print()
print(f"Outputs: {out_dir}/")

DIMER Document-Type Classification Notebook
---------------------------------------------
Sample: 320 balanced RVL-CDIP-derived pages
Classes: 16

Random expectation: 0.0625
Majority baseline:  0.0625
Accuracy:           0.9531
Macro F1:           0.9530
Top-3 accuracy:     0.9938

Source checkpoint SHA-256: 1b7a901642c36ec7e32de997683223faf02028624fb2e62a7e7e798a5dd1344e
Converted SafeTensors SHA-256: b9484ea5054bf02ede62493b4e7789a40873a513fb4a94c5aa786869d841d1f7
Conversion parity max abs logit diff: 0.000e+00

Robustness:
  original         accuracy=0.812 stability=1.000
  rotate90         accuracy=0.500 stability=0.625
  rotate180        accuracy=0.688 stability=0.875
  low_resolution   accuracy=0.750 stability=0.938
  center_crop      accuracy=0.812 stability=1.000

Outputs: outputs/document_type_classification/


## Try it yourself — one controlled change

Use the existing Section 16 variants without rerunning Configuration or changing the original evaluation.

1. **Predict:** before revealing the Section 16 table, predict whether `rotate90` or `low_resolution` will reduce accuracy more on the fixed robustness pages, and explain which page cues each changes.
2. **Change one thing:** choose `rotate90` as your comparison with `original`. Both rows use identical selected pages and model weights; only the image transformation differs. Keep the canonical controls unchanged.
3. **Run:** if robustness is enabled (the default), execute Section 16 once. If it was disabled, use a separate notebook copy and fresh runtime with `RUN_ROBUSTNESS=True`; do not replace the original run's exports. No new code or tuning is required.
4. **Observe:** read original and rotate90 accuracy, prediction stability, and mean gold score. Record the difference (rotated minus original) and a page whose class changed. As a separate comparison, repeat the reading for low_resolution against original, holding everything else fixed.
5. **Explain:** "I predicted __. Rotation changed accuracy from __ to __ on the same 16 pages; stability was __. This suggests __ about layout cues, but one page per class and a previously inspected test sample limit the conclusion."

This is exploratory robustness evidence, not a new independent test or a reason to tune the model on these pages. Canonical prediction arrays and reports are unchanged by reading these comparisons.


## Self-paced checkpoint

Before opening the sample interpretation, answer:

1. What did the model/system receive as input, and what did it produce?
2. Which baseline/reference tells you whether the learned model added value?
3. What failure mode or tradeoff matters most here?
4. What additional evidence would you want before transferring the result to a new domain?

<details>
<summary><b>Show a sample interpretation</b></summary>

Document-type classification answers 'what kind of page is this?' rather than reading its text or answering a question. DiT still learns from visual/textual appearance encoded in pixels, its class vocabulary is fixed, and confidence scores should not be treated as calibrated certainty.

Use the outputs from **your run** when writing your final answer; small numeric differences across supported runtimes are possible.

</details>


## Write an evidence-based conclusion

1. **State the question** tested by this notebook.
2. **Report the primary result** against the relevant baseline/reference.
3. **Add supporting evidence** from a secondary metric, error pattern, disagreement, or qualitative diagnostic.
4. **Account for cost/complexity** when it materially affects the comparison.
5. **State the limits** of the data, split, model revision, and configuration.

Report the held-out classification result against the majority/random references, name the most important confusion or robustness failure mode, and state the fixed-vocabulary and source-dataset limitations.


# Troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| Accelerator unavailable or execution is unexpectedly slow | The runtime does not match the documented resource envelope | Select the documented accelerator/runtime, start a fresh session, and run top-to-bottom. |
| Package/version or stale-module error | Incompatible libraries were already imported in the hosted kernel | Start a fresh runtime and choose **Run all** before importing extra packages. Do not bypass version checks. |
| Model/sample digest or byte-size check fails | Download is incomplete or upstream bytes differ from the pinned artifact | Remove the affected runtime cache/download and rerun. Do not disable the integrity check. |
| Out-of-memory or runtime restart | Too many large models/intermediates are resident | Use the default tier, follow explicit unload/release steps, and avoid combining optional heavy branches. |
| BYOD validation fails | Input does not satisfy the documented schema, shape, labels, or limits | Follow the validation message, correct the indicated field/format, then rerun the BYOD branch. |
| Your numbers differ slightly | Supported hardware/library execution can introduce small numerical variation | Verify the split, model revision, metric definition, and qualitative pattern before treating the difference as substantive. |


# Glossary

| Term | Meaning in this notebook |
|---|---|
| **Document-type classification** | Assigning a whole document page to one label such as a document category. |
| **Fixed vocabulary** | The predefined set of classes the classifier can output. |
| **Confusion matrix** | Counts of true classes versus predicted classes. |
| **Confidence margin** | The difference between leading class scores; useful as a relative ambiguity indicator, not calibrated certainty. |
| **Selective classification** | Withholding some low-confidence predictions according to a chosen rule. |
| **Robustness probe** | A controlled input change used to test sensitivity without changing the underlying label. |